In [ ]:
import torch
import testdata
# note: had to move this notebook and testdata.py into 
# the multicor_fa directory to run
from _em import _EM_step_no_private_stable, fit_EM_iter

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

Generate fake data

In [ ]:
params = {
    'd': 15, 
    'k': [0, 0, 0], 
    'p': [15, 13, 8], 
    'n': 5000,
    'sigsq': [0.3, 0.7, 0.5]
}

No private factors, so Y = WZ + E


In [32]:
def run_test(params, n_iter=1000, data='complete'):
    metrics = {
        'WWt_corr': [],
        'Phi_corr': [],
        'Sigma_corr': [] # WW^T + Phi
    }
    
    for i in range(n_iter):
        if (i+1) % (0.2*n_iter) == 0:
            print(f"{i+1} simulations completed")

        # generate data
        Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
        Y = Y.T
    
        if data=='missing_disjoint':
            Y[1000:1025, :params['p'][0]] = float('nan')
            Y[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
            Y[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')
            
        elif data=='missing_overlap':
            Y[1000:1050, :params['p'][0]] = float('nan')
            Y[1040:1100, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
            Y[1075:1150, params['p'][0]+params['p'][1]:] = float('nan')

        W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
        Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

        # ground truths
        WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
        P = torch.cat([Phi.flatten() for Phi in Phi_true])
        Sigma_true = torch.cat([
            torch.flatten((W_true[i] @ W_true[i].T) + Phi_true[i]) 
            for i in range(len(W_true))
        ])
    
        # run EM
        W_new, _, Phi_new, _, _ = fit_EM_iter(
            Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=(data!='complete')
        )
        WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
        P_test = torch.cat([Phi.flatten() for Phi in Phi_new])
        Sigma_test = torch.cat([
            torch.flatten((W_new[i] @ W_new[i].T) + Phi_new[i]) 
            for i in range(len(W_new))
        ])
        
        # get metrics
        metrics['WWt_corr'].append(torch.corrcoef(
            torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
        )[0,1].item())
        metrics['Phi_corr'].append(torch.corrcoef(
            torch.stack([P, P_test], dim=0)
        )[0,1].item())
        metrics['Sigma_corr'].append(torch.corrcoef(
            torch.stack([Sigma_true, Sigma_test], dim=0)
        )[0,1].item())
        
    print('\nMetrics:')
    for k, v in metrics.items():
        print(' ', k)
        v = torch.tensor(v)
        print(f"\tMean: {round(torch.mean(v).item(), 4)}")
        print(f"\tMin: {round(torch.min(v).item(), 4)}")
        print(f"\tMax: {round(torch.max(v).item(), 4)}")

Attempt at more systematic testing for complete data case

In [33]:
run_test(params)

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed

Metrics:
  WWt_corr
	Mean: 0.9962
	Min: 0.9833
	Max: 0.9987
  Phi_corr
	Mean: 0.9359
	Min: 0.5691
	Max: 0.9824
  Sigma_corr
	Mean: 0.9977
	Min: 0.9889
	Max: 0.9992


Attempt at more systematic testing for missing data case

In [34]:
run_test(params, data='missing_disjoint')

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed

Metrics:
  WWt_corr
	Mean: 0.9976
	Min: 0.9938
	Max: 0.9989
  Phi_corr
	Mean: 0.9265
	Min: 0.6083
	Max: 0.9771
  Sigma_corr
	Mean: 0.9987
	Min: 0.9961
	Max: 0.9995


Missing data with more than one mode missing in some samples

In [35]:
run_test(params, data='missing_overlap')

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed

Metrics:
  WWt_corr
	Mean: 0.9979
	Min: 0.9951
	Max: 0.9989
  Phi_corr
	Mean: 0.9196
	Min: 0.7332
	Max: 0.9721
  Sigma_corr
	Mean: 0.9988
	Min: 0.9968
	Max: 0.9995
